# Stock risk report
inmeng Pan · HW2

Real observations from the supplied homework database, using `ohlc_hw_cleaned`.
Requested and actual observation coverage are shown separately. Adjusted close drives
report returns and drawdown; missing sessions relative to this ticker's reference
`ohlc` keys are retained as NaN. Source histories ending early are not extrapolated.


In [ ]:
ticker = "AAPL"
start_date = "2024-06-01"
end_date = "2024-11-30"
db_path = "data/data.db"


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, JSON
from analysis import (load_stock, calculate_returns, calculate_volatility,
    calculate_max_drawdown, calculate_sharpe_ratio, stock_metrics, plot_stock, report_series)

data = load_stock(db_path, ticker, start_date, end_date)
print(f"{ticker} from {start_date} to {end_date}: {len(data)} trading days")
print(f"Actual observations: {data.ts.min().date()} to {data.ts.max().date()}")
if len(data) < 21:
    print("Insufficient history for 20-return rolling volatility: at least 21 prices required.")
if len(data) < 50:
    print("Insufficient history for the 50-price moving average.")
display(data.head(), data.tail())

prices, daily_returns, rolling_volatility, drawdown = report_series(data)
print("Calendar sessions:", len(prices), "| Missing observations:", int(prices.isna().sum()),
      "| Valid one-session returns:", int(daily_returns.notna().sum()))
if prices.isna().any():
    print("Gaps remain missing. Gap-spanning price moves are excluded from daily statistics.")


## Daily risk series
Log returns drive volatility, Sharpe and daily summary statistics. Rolling volatility
uses 20 nonmissing returns and the sample standard deviation; warm-up values remain
missing. Drawdown can recover, while its minimum is the maximum loss from a prior peak.
The 2% risk-free rate is the assignment's annual Sharpe approximation.


Internal calendar gaps remain NaN. Only returns between adjacent supplied sessions enter daily statistics. Drawdown uses observed peaks and may miss an unobserved intragap loss.


In [ ]:
data["log_return"] = data.ts.map(daily_returns)
data["rolling_volatility"] = data.ts.map(rolling_volatility)
data["drawdown"] = data.ts.map(drawdown)
display(data[["ts", "adj_close", "log_return", "rolling_volatility", "drawdown"]].tail())
fig = plot_stock(data, ticker, start_date, end_date)
display(fig)
plt.close(fig)


## Summary
Percentages are formatting only; the exported metrics remain fractions. Annualized
return compounds the observed log returns at 252 periods per year. Whole-period
volatility differs from the batch comparison's mean rolling volatility. Positive days
include zero-return days in the denominator. Undefined estimates display as N/A.


With missing sessions, annualized return summarizes available one-session returns, whereas total return uses the first/last observed prices. They need not compound to the same result. Neither gap returns nor zero returns are invented.


In [ ]:
report_metrics = stock_metrics(data)
rows = [
    ("Total Return", "total_return"), ("Annualized Return", "annualized_return"),
    ("Annualized Volatility", "annualized_volatility"), ("Sharpe Ratio", "sharpe_ratio"),
    ("Maximum Drawdown", "max_drawdown"), ("Best Day", "best_day"),
    ("Worst Day", "worst_day"), ("Positive Days", "positive_days"),
    ("Average Up Day", "average_up_day"), ("Average Down Day", "average_down_day")]
formatted = []
for label, key in rows:
    value = report_metrics[key]
    formatted.append((label, "N/A" if pd.isna(value) else f"{value:.2f}" if key == "sharpe_ratio" else f"{value:+.2%}"))
display(pd.DataFrame(formatted, columns=["Metric", "Value"]).set_index("Metric"))
json_metrics = {key: None if isinstance(value, float) and not np.isfinite(value) else value
                for key, value in report_metrics.items()}
display(JSON({"report_metrics": json_metrics}))
